In [ ]:
# Figure 5H: Neurotransmitter-specific presynaptic-arbor proximity

from pathlib import Path
from collections import deque
from itertools import combinations
import io
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import kruskal, mannwhitneyu
import open3d as o3d

data_dir = Path('data')
data = h5py.File(data_dir / 'fig5h.h5', 'r')
out_dir = Path('fig5_outputs')
out_dir.mkdir(parents=True, exist_ok=True)

nt_order = ['ACH', 'GABA', 'GLUT', 'DA', 'SER', 'OCT']
nt_display = {'ACH': 'ACh', 'GABA': 'GABA', 'GLUT': 'GLUT', 'DA': 'DA', 'SER': 'SER', 'OCT': 'OCT'}
nt_colors = {'ACH': '#2020df', 'GABA': '#df2020', 'GLUT': '#df9c20', 'DA': '#963939', 'SER': '#701070', 'OCT': '#107010'}

excluded_neuropils = {'LA_L', 'LA_R'}
max_neurons_per_nt = 250
min_contact_rows_per_neuron = 20
min_unique_sites_per_neuron = 20
cable_spacing_nm = 250.0
max_site_to_cable_um = 5.0
surface_query_chunk = 100000
run_whole_neuron_sensitivity = True
hist_bin_um = 0.25
hist_max_um = 200.0
proximity_thresholds_um = [2.0, 5.0, 10.0, 20.0]
n_bootstrap = 5000
random_seed = 20260913

plt.rcParams.update({'font.family': 'Arial', 'font.size': 9, 'axes.labelsize': 9.5, 'axes.titlesize': 10, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8, 'axes.linewidth': 0.85, 'xtick.direction': 'out', 'ytick.direction': 'out', 'xtick.major.width': 0.85, 'ytick.major.width': 0.85, 'xtick.major.size': 3.5, 'ytick.major.size': 3.5, 'svg.fonttype': 'none', 'pdf.fonttype': 42, 'ps.fonttype': 42})

def exact_root(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.endswith('.0'):
        text = text[:-2]
    return text

def normalize_nt(values):
    x = values.astype('string').str.strip().str.upper()
    return x.replace({'ACETYLCHOLINE': 'ACH', 'GLUTAMATE': 'GLUT', 'DOPAMINE': 'DA', 'SEROTONIN': 'SER', '5HT': 'SER', '5-HT': 'SER', 'OCTOPAMINE': 'OCT'})

def bh_fdr(pvalues):
    pvalues = np.asarray(pvalues, dtype=np.float64)
    result = np.full(len(pvalues), np.nan, dtype=np.float64)
    finite = np.isfinite(pvalues)
    p = pvalues[finite]
    order = np.argsort(p)
    ranked = p[order]
    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1, dtype=np.float64)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    back = np.empty_like(adjusted)
    back[order] = adjusted
    result[finite] = back
    return result

def weighted_quantile(values, weights, q):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[good]
    weights = weights[good]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights)
    target = float(q) * cumulative[-1]
    index = int(np.searchsorted(cumulative, target, side='left'))
    index = min(index, len(values) - 1)
    return float(values[index])

def weighted_mean(values, weights):
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    return float(np.average(values[good], weights=weights[good]))

def bootstrap_median_ci(values, n_boot=n_bootstrap, seed=random_seed):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = len(values)
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        index = rng.integers(0, n, size=n)
        boot[i] = np.median(values[index])
    return tuple(np.quantile(boot, [0.025, 0.975]))

# NT-balanced neuron cohort
selected = pd.read_csv(io.BytesIO(data['tables/neurons'][:].tobytes()), dtype={'pre_root_id': 'string'}, low_memory=False)
selected['pre_root_id'] = selected['pre_root_id'].map(exact_root).astype('string')
selected['nt_type'] = normalize_nt(selected['nt_type'])
selected['nt_type_score'] = pd.to_numeric(selected['nt_type_score'], errors='coerce')
selected['contact_row_count'] = pd.to_numeric(selected['contact_row_count'], errors='coerce')
selected = selected.loc[selected['nt_type'].isin(nt_order) & selected['nt_type_score'].ge(0.5) & selected['contact_row_count'].ge(min_contact_rows_per_neuron)].copy()
sites = pd.read_csv(io.BytesIO(data['tables/sites'][:].tobytes()), compression='gzip', dtype={'pre_root_id': 'string', 'neuropil': 'string'}, low_memory=False)
sites['pre_root_id'] = sites['pre_root_id'].map(exact_root).astype('string')
sites = sites.loc[~sites['neuropil'].isin(excluded_neuropils)].copy()
for column in ['x_nm', 'y_nm', 'z_nm']:
    sites[column] = pd.to_numeric(sites[column], errors='coerce')
sites = sites.dropna(subset=['pre_root_id', 'x_nm', 'y_nm', 'z_nm']).copy()
site_counts = sites.groupby('pre_root_id').size().rename('unique_presynaptic_site_count').reset_index()
selected = selected.merge(site_counts, on='pre_root_id', how='left', validate='one_to_one')
selected['unique_presynaptic_site_count'] = selected['unique_presynaptic_site_count'].fillna(0).astype(int)
selected = selected.loc[selected['unique_presynaptic_site_count'] >= min_unique_sites_per_neuron].copy()
selection_parts = []
for nt_index, nt in enumerate(nt_order):
    group = selected.loc[selected['nt_type'] == nt].copy().sort_values('pre_root_id').reset_index(drop=True)
    if len(group) <= max_neurons_per_nt:
        chosen = group.copy()
    else:
        rng = np.random.default_rng(random_seed + nt_index * 1009)
        index = np.sort(rng.choice(len(group), size=max_neurons_per_nt, replace=False))
        chosen = group.iloc[index].copy().reset_index(drop=True)
    selection_parts.append(chosen)
analysis_neurons = pd.concat(selection_parts, ignore_index=True).sort_values(['nt_type', 'pre_root_id']).reset_index(drop=True)
print(analysis_neurons['nt_type'].value_counts().reindex(nt_order))
analysis_root_set = set(analysis_neurons['pre_root_id'].astype(str))
sites = sites.loc[sites['pre_root_id'].astype(str).isin(analysis_root_set)].copy()
site_groups = {str(root): group.reset_index(drop=True) for root, group in sites.groupby('pre_root_id', sort=False)}

# Tracheole surface and presynaptic-arbor reconstruction
def build_trachea_scene():
    scene = o3d.t.geometry.RaycastingScene(nthreads=0)
    for mesh in data['tracheoles'].values():
        vertices = np.asarray(mesh['vertices'][:], dtype=np.float32)
        faces = np.asarray(mesh['faces'][:], dtype=np.uint32)
        triangle_mesh = o3d.t.geometry.TriangleMesh(o3d.core.Tensor(vertices, dtype=o3d.core.Dtype.Float32), o3d.core.Tensor(faces, dtype=o3d.core.Dtype.UInt32))
        scene.add_triangles(triangle_mesh)
    return scene
scene = build_trachea_scene()

def load_skeleton_nodes(root):
    nodes = pd.read_csv(io.BytesIO(data[f'skeletons/{root}'][:].tobytes()), compression='gzip', low_memory=False)
    for column in ['node_id', 'parent_id', 'x_nm', 'y_nm', 'z_nm']:
        nodes[column] = pd.to_numeric(nodes[column], errors='coerce')
    nodes = nodes.dropna(subset=['node_id', 'x_nm', 'y_nm', 'z_nm']).copy()
    nodes['node_id'] = nodes['node_id'].astype(np.int64)
    return nodes

def densify_skeleton_edges(nodes, max_spacing_nm):
    lookup = nodes.set_index('node_id')[['x_nm', 'y_nm', 'z_nm']]
    point_parts = []
    weight_parts = []
    child_parts = []
    parent_parts = []
    for row in nodes.itertuples(index=False):
        if pd.isna(row.parent_id):
            continue
        child_id = int(row.node_id)
        parent_id = int(row.parent_id)
        if parent_id not in lookup.index:
            continue
        a = lookup.loc[parent_id].to_numpy(dtype=np.float64)
        b = np.asarray([row.x_nm, row.y_nm, row.z_nm], dtype=np.float64)
        vector = b - a
        length_nm = float(np.linalg.norm(vector))
        if not np.isfinite(length_nm) or length_nm <= 0:
            continue
        n = max(1, int(np.ceil(length_nm / max_spacing_nm)))
        fractions = (np.arange(n, dtype=np.float64) + 0.5) / n
        points = a[None, :] + fractions[:, None] * vector[None, :]
        subsegment_length_nm = length_nm / n
        point_parts.append(points.astype(np.float32))
        weight_parts.append(np.full(n, subsegment_length_nm, dtype=np.float32))
        child_parts.append(np.full(n, child_id, dtype=np.int64))
        parent_parts.append(np.full(n, parent_id, dtype=np.int64))
    return (np.vstack(point_parts), np.concatenate(weight_parts), np.concatenate(child_parts), np.concatenate(parent_parts))

def presynaptic_arbor_mask(nodes, cable_points, child_ids, parent_ids, site_xyz_nm):
    cable_tree = cKDTree(cable_points)
    nearest_distance_nm, nearest_index = cable_tree.query(site_xyz_nm, k=1, workers=-1)
    aligned = nearest_distance_nm <= max_site_to_cable_um * 1000.0
    n_sites = int(len(site_xyz_nm))
    n_aligned = int(np.sum(aligned))
    alignment_fraction = n_aligned / n_sites if n_sites else np.nan
    if n_aligned < 2:
        return (np.zeros(len(cable_points), dtype=bool), n_sites, n_aligned, alignment_fraction)
    mapped_indices = nearest_index[aligned]
    marked_nodes = set(child_ids[mapped_indices].astype(int))
    marked_nodes.update(parent_ids[mapped_indices].astype(int))
    adjacency = {}
    for child_id, parent_id in zip(child_ids, parent_ids):
        child_id = int(child_id)
        parent_id = int(parent_id)
        adjacency.setdefault(child_id, set()).add(parent_id)
        adjacency.setdefault(parent_id, set()).add(child_id)
    active = set(adjacency)
    degree = {node: len(adjacency[node]) for node in active}
    queue = deque([node for node in active if degree[node] <= 1 and node not in marked_nodes])
    while queue:
        node = queue.popleft()
        if node not in active:
            continue
        if node in marked_nodes:
            continue
        if degree[node] > 1:
            continue
        active.remove(node)
        for neighbor in adjacency[node]:
            if neighbor not in active:
                continue
            degree[neighbor] -= 1
            if degree[neighbor] <= 1 and neighbor not in marked_nodes:
                queue.append(neighbor)
    mask = np.isin(child_ids, list(active)) & np.isin(parent_ids, list(active))
    return (mask, n_sites, n_aligned, alignment_fraction)

def trachea_surface_distance_um(points_nm):
    points_nm = np.asarray(points_nm, dtype=np.float32)
    distance_um = np.empty(len(points_nm), dtype=np.float32)
    for start in range(0, len(points_nm), surface_query_chunk):
        stop = min(start + surface_query_chunk, len(points_nm))
        points = points_nm[start:stop]
        answer = scene.compute_closest_points(o3d.core.Tensor(points, dtype=o3d.core.Dtype.Float32), nthreads=0)
        closest = answer['points'].numpy()
        distance_um[start:stop] = np.linalg.norm(points - closest, axis=1) / 1000.0
    return distance_um
hist_edges = np.arange(0.0, hist_max_um + hist_bin_um, hist_bin_um, dtype=np.float64)
n_hist_bins = len(hist_edges) - 1

def weighted_histogram(distance_um, weights_um):
    distance_um = np.asarray(distance_um, dtype=np.float64)
    weights_um = np.asarray(weights_um, dtype=np.float64)
    good = np.isfinite(distance_um) & np.isfinite(weights_um) & (weights_um > 0)
    distance_um = distance_um[good]
    weights_um = weights_um[good]
    overflow_length_um = float(np.sum(weights_um[distance_um >= hist_max_um]))
    within = distance_um < hist_max_um
    hist = np.histogram(distance_um[within], bins=hist_edges, weights=weights_um[within])[0].astype(np.float64)
    return (hist, overflow_length_um)

def summarize_cable(root, nt_type, mode, distance_um, weights_nm, n_sites, n_sites_aligned, site_alignment_fraction):
    distance_um = np.asarray(distance_um, dtype=np.float64)
    weights_um = np.asarray(weights_nm, dtype=np.float64) / 1000.0
    good = np.isfinite(distance_um) & np.isfinite(weights_um) & (weights_um > 0)
    distance_um = distance_um[good]
    weights_um = weights_um[good]
    q25 = weighted_quantile(distance_um, weights_um, 0.25)
    median = weighted_quantile(distance_um, weights_um, 0.5)
    q75 = weighted_quantile(distance_um, weights_um, 0.75)
    r90 = weighted_quantile(distance_um, weights_um, 0.9)
    row = {
        'pre_root_id': str(root), 'nt_type': nt_type, 'cable_mode': mode,
        'n_presynaptic_sites': int(n_sites), 'n_sites_aligned_to_cable': int(n_sites_aligned), 'site_alignment_fraction': float(site_alignment_fraction),
        'n_cable_samples': int(len(distance_um)), 'cable_length_um': float(np.sum(weights_um)), 'mean_distance_um': weighted_mean(distance_um, weights_um),
        'q25_distance_um': q25, 'median_distance_um': median, 'q75_distance_um': q75,
        'r90_distance_um': r90
    }
    for threshold in proximity_thresholds_um:
        row[f'fraction_cable_within_{threshold:g}um'] = float(np.sum(weights_um[distance_um <= threshold]) / np.sum(weights_um))
    hist, overflow_length_um = weighted_histogram(distance_um, weights_um)
    row['histogram_overflow_length_um'] = overflow_length_um
    return (row, hist)

# Cable distances and neuron-level summaries
summary_rows = []
hist_accumulator = {(mode, nt): np.zeros(n_hist_bins) for mode in (['presynaptic_arbor', 'whole_neuron'] if run_whole_neuron_sensitivity else ['presynaptic_arbor']) for nt in nt_order}
for neuron in analysis_neurons.itertuples(index=False):
    root, nt_type = (str(neuron.pre_root_id), str(neuron.nt_type))
    site_group = site_groups[root]
    nodes = load_skeleton_nodes(root)
    cable_points, cable_weights_nm, child_ids, parent_ids = densify_skeleton_edges(nodes, cable_spacing_nm)
    site_xyz_nm = site_group[['x_nm', 'y_nm', 'z_nm']].to_numpy(dtype=np.float32)
    arbor_mask, n_sites, n_aligned, alignment = presynaptic_arbor_mask(nodes, cable_points, child_ids, parent_ids, site_xyz_nm)
    arbor_distances = trachea_surface_distance_um(cable_points[arbor_mask])
    arbor_row, arbor_hist = summarize_cable(root, nt_type, 'presynaptic_arbor', arbor_distances, cable_weights_nm[arbor_mask], n_sites, n_aligned, alignment)
    rows = [arbor_row]
    hist_accumulator['presynaptic_arbor', nt_type] += arbor_hist
    if run_whole_neuron_sensitivity:
        whole_distances = trachea_surface_distance_um(cable_points)
        whole_row, whole_hist = summarize_cable(root, nt_type, 'whole_neuron', whole_distances, cable_weights_nm, n_sites, n_aligned, alignment)
        rows.append(whole_row)
        hist_accumulator['whole_neuron', nt_type] += whole_hist
    summary_rows.extend(rows)
per_neuron = pd.DataFrame(summary_rows)
per_neuron['pre_root_id'] = per_neuron['pre_root_id'].astype('string')
print(per_neuron.loc[per_neuron['cable_mode'] == 'presynaptic_arbor'].groupby('nt_type')['pre_root_id'].nunique().reindex(nt_order))
def rank_biserial_from_u(u, n_a, n_b):
    return float(2.0 * u / (n_a * n_b) - 1.0)

# Neuron-level statistics
def analyze_mode(dataframe, mode):
    data = dataframe.loc[dataframe['cable_mode'] == mode].copy()
    data['median_distance_um'] = pd.to_numeric(data['median_distance_um'], errors='coerce')
    data = data.dropna(subset=['median_distance_um', 'nt_type'])
    summary_rows = []
    for nt_index, nt in enumerate(nt_order):
        group = data.loc[data['nt_type'] == nt]
        values = group['median_distance_um'].to_numpy(dtype=np.float64)
        if len(values) == 0:
            continue
        ci_low, ci_high = bootstrap_median_ci(values, seed=random_seed + nt_index * 1009)
        row = {'cable_mode': mode, 'nt_type': nt, 'n_neurons': int(len(values)), 'median_neuron_median_distance_um': float(np.median(values)), 'q25_neuron_median_distance_um': float(np.quantile(values, 0.25)), 'q75_neuron_median_distance_um': float(np.quantile(values, 0.75)), 'bootstrap_ci95_low_um': float(ci_low), 'bootstrap_ci95_high_um': float(ci_high), 'median_neuron_mean_distance_um': float(np.median(group['mean_distance_um'])), 'median_cable_length_um': float(np.median(group['cable_length_um']))}
        for threshold in proximity_thresholds_um:
            column = f'fraction_cable_within_{threshold:g}um'
            row[f'median_{column}'] = float(np.median(group[column]))
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    groups = []
    group_labels = []
    for nt in nt_order:
        values = data.loc[data['nt_type'] == nt, 'median_distance_um'].to_numpy(dtype=np.float64)
        values = values[np.isfinite(values)]
        if len(values):
            groups.append(values)
            group_labels.append(nt)
    if len(groups) >= 2:
        h, p_kw = kruskal(*groups)
        n_total = int(sum((len(values) for values in groups)))
        k = len(groups)
        epsilon_squared = max(0.0, (h - k + 1) / (n_total - k))
    else:
        h = np.nan
        p_kw = np.nan
        epsilon_squared = np.nan
        n_total = int(sum((len(values) for values in groups)))
        k = len(groups)
    omnibus = pd.DataFrame([{
        'cable_mode': mode, 'n_neurons': n_total, 'n_nt_groups': k,
        'kruskal_H': float(h), 'degrees_of_freedom': k - 1 if k else np.nan, 'P': float(p_kw),
        'epsilon_squared': float(epsilon_squared)
    }])
    pair_rows = []
    for nt_a, nt_b in combinations(group_labels, 2):
        a = data.loc[data['nt_type'] == nt_a, 'median_distance_um'].to_numpy(dtype=np.float64)
        b = data.loc[data['nt_type'] == nt_b, 'median_distance_um'].to_numpy(dtype=np.float64)
        u, p = mannwhitneyu(a, b, alternative='two-sided', method='auto')
        pair_rows.append({
            'cable_mode': mode, 'nt_1': nt_a, 'nt_2': nt_b,
            'n_1': int(len(a)), 'n_2': int(len(b)), 'median_1_um': float(np.median(a)),
            'median_2_um': float(np.median(b)), 'median_difference_1_minus_2_um': float(np.median(a) - np.median(b)), 'U': float(u),
            'rank_biserial_1_vs_2': rank_biserial_from_u(u, len(a), len(b)), 'P_raw': float(p)
        })
    pairwise = pd.DataFrame(pair_rows)
    if len(pairwise):
        pairwise['q_BH_FDR'] = bh_fdr(pairwise['P_raw'].to_numpy(dtype=np.float64))
    return (summary, omnibus, pairwise)
all_summary_parts = []
all_omnibus_parts = []
all_pairwise_parts = []
modes_to_analyze = ['presynaptic_arbor']
if run_whole_neuron_sensitivity:
    modes_to_analyze.append('whole_neuron')
for mode in modes_to_analyze:
    mode_summary, mode_omnibus, mode_pairwise = analyze_mode(per_neuron, mode)
    all_summary_parts.append(mode_summary)
    all_omnibus_parts.append(mode_omnibus)
    all_pairwise_parts.append(mode_pairwise)
nt_summary = pd.concat(all_summary_parts, ignore_index=True)
omnibus_df = pd.concat(all_omnibus_parts, ignore_index=True)
pairwise_df = pd.concat(all_pairwise_parts, ignore_index=True)

# Cable-length-weighted ECDF and neuron-level distances
def plot_mode(mode):
    data = per_neuron.loc[per_neuron['cable_mode'] == mode].copy()
    fig = plt.figure(figsize=(7.25, 3.35), facecolor='white')
    gs = fig.add_gridspec(1, 2, width_ratios=[1.08, 0.92], wspace=0.34)
    ax = fig.add_subplot(gs[0, 0])
    for nt in nt_order:
        hist = hist_accumulator.get((mode, nt))
        if hist is None or np.sum(hist) <= 0:
            continue
        cumulative = np.cumsum(hist) / np.sum(hist)
        x = hist_edges[1:]
        ax.step(x, cumulative, where='post', color=nt_colors[nt], linewidth=1.55, label=nt_display[nt])
    ax.set_xlabel('Cable-to-trachea distance (µm)')
    ax.set_ylabel('Cumulative cable length')
    ax.set_ylim(0, 1.01)
    ax.set_xlim(0, 30)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, ncol=2, loc='lower right')
    ax.text(-0.16, 1.03, 'A', transform=ax.transAxes, fontsize=12, ha='left', va='bottom')
    ax = fig.add_subplot(gs[0, 1])
    jitter_rng = np.random.default_rng(random_seed)
    for x_index, nt in enumerate(nt_order):
        values = data.loc[data['nt_type'] == nt, 'median_distance_um'].to_numpy(dtype=np.float64)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            continue
        jitter = jitter_rng.uniform(-0.16, 0.16, size=len(values))
        ax.scatter(np.full(len(values), x_index, dtype=np.float64) + jitter, values, s=11, color=nt_colors[nt], alpha=0.34, linewidths=0, rasterized=True, zorder=1)
        q25 = float(np.quantile(values, 0.25))
        median = float(np.median(values))
        q75 = float(np.quantile(values, 0.75))
        ax.plot([x_index, x_index], [q25, q75], color='0.15', linewidth=1.8, zorder=3)
        ax.scatter(x_index, median, s=34, facecolor='white', edgecolor='black', linewidth=0.85, zorder=4)
    ax.set_xticks(np.arange(len(nt_order)))
    ax.set_xticklabels([nt_display[nt] for nt in nt_order], rotation=35, ha='right')
    ax.set_ylabel('Median cable-to-trachea distance (µm)')
    ax.set_xlabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.text(-0.19, 1.03, 'B', transform=ax.transAxes, fontsize=12, ha='left', va='bottom')
    fig.subplots_adjust(left=0.095, right=0.985, bottom=0.21, top=0.96)
    fig.savefig(out_dir / 'fig5h.svg', bbox_inches='tight')
    plt.show()
plot_mode('presynaptic_arbor')
print(nt_summary[['cable_mode', 'nt_type', 'n_neurons', 'median_neuron_median_distance_um', 'bootstrap_ci95_low_um', 'bootstrap_ci95_high_um']].to_string(index=False))
print(omnibus_df.to_string(index=False))